# Judge Baseline Training (Classification)

This notebook trains the classification baseline used for comparison against the main regression judge. It predicts one of the eight target rating groups directly.

**Input:** `data/imdb_sup.csv`

**Outputs:** `artifacts/judge_classification_checkpoints/`, `artifacts/judge_classification_model/`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

PROJECT_ROOT = next(
    (path.resolve() for path in (Path.cwd(), Path.cwd().parent) if (path / "data" / "imdb_sup.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/imdb_sup.csv. Run the notebook from the repository root or notebooks/ directory."
    )

DATA_PATH = PROJECT_ROOT / 'data' / 'imdb_sup.csv'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
CHECKPOINT_DIR = ARTIFACTS_DIR / 'judge_classification_checkpoints'
MODEL_DIR = ARTIFACTS_DIR / 'judge_classification_model'
LOG_DIR = ARTIFACTS_DIR / 'logs'

for path in (ARTIFACTS_DIR, CHECKPOINT_DIR, MODEL_DIR, LOG_DIR):
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH, on_bad_lines='skip', engine='python')

target_ratings = [1, 2, 3, 4, 7, 8, 9, 10]
df = df[df['Rating'].isin(target_ratings)].copy()

label_map = {1: 0, 2: 1, 3: 2, 4: 3, 7: 4, 8: 5, 9: 6, 10: 7}
df['label'] = df['Rating'].map(label_map)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['Review'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
)

print(f"Loaded {len(df):,} filtered reviews from {DATA_PATH}")
print(f"Training examples: {len(train_texts):,}")
print(f"Validation examples: {len(val_texts):,}")


In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=256)

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=8,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Model is running on: {device}")


In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall,
    }

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir=str(LOG_DIR),
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='loss',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    compute_metrics=compute_metrics,
)

trainer.train()


In [ ]:
print('Generating predictions for confusion matrix...')
predictions = trainer.predict(val_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

class_names = [1, 2, 3, 4, 7, 8, 9, 10]
cm = confusion_matrix(true_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(10, 10))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Confusion Matrix: True Ratings vs. Predicted Ratings')
plt.show()


In [ ]:
model.save_pretrained(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

print(f'The classification baseline is ready and saved to {MODEL_DIR}')
